In [1]:
import pandas as pd

df = pd.read_csv('data/flights_log.csv')
df.shape

(127, 11)

In [2]:
df.head()

,collected_at,route,flight_date,flight_status,dep_scheduled,dep_actual,dep_delay_min,arr_scheduled,arr_actual,arr_delay_min,airline
0,2026-07-08T22:10:30.051593,BOM-FRA,2026-07-09,scheduled,2026-07-09T02:40:00+00:00,NaN,NaN,2026-07-09T08:05:00+00:00,NaN,20.0,Lufthansa
1,2026-07-08T22:10:30.051661,BOM-FRA,2026-07-08,landed,2026-07-08T11:40:00+00:00,2026-07-08T12:10:00+00:00,NaN,2026-07-08T17:55:00+00:00,2026-07-08T17:30:00+00:00,NaN,Lufthansa
2,2026-07-08T22:10:30.051677,BOM-FRA,2026-07-08,landed,2026-07-08T11:40:00+00:00,2026-07-08T12:10:00+00:00,NaN,2026-07-08T17:55:00+00:00,2026-07-08T17:30:00+00:00,NaN,Air India
3,2026-07-08T22:10:30.051687,BOM-FRA,2026-07-08,landed,2026-07-08T08:10:00+00:00,2026-07-08T08:34:00+00:00,9.0,2026-07-08T13:30:00+00:00,2026-07-08T13:39:00+00:00,9.0,DHL Air
4,2026-07-08T22:10:30.051696,BOM-FRA,2026-07-08,scheduled,2026-07-08T02:40:00+00:00,2026-07-08T04:08:00+00:00,74.0,2026-07-08T08:05:00+00:00,NaN,74.0,Air India


In [3]:
df['flight_status'].value_counts()

flight_status
landed       57
scheduled    41
active       29
Name: count, dtype: int64

In [4]:
landed_flights = df[df['flight_status'] == 'landed']
landed_flights['dep_delay_min'].describe()

count     8.000000
mean     19.625000
std       9.425459
min       9.000000
25%       9.000000
50%      23.000000
75%      28.000000
max      28.000000
Name: dep_delay_min, dtype: float64

In [5]:
landed_flights['dep_delay_min'].isna().sum()

np.int64(49)

In [6]:
landed_flights[['dep_scheduled', 'dep_actual', 'dep_delay_min']].head(10)

,dep_scheduled,dep_actual,dep_delay_min
1,2026-07-08T11:40:00+00:00,2026-07-08T12:10:00+00:00,NaN
2,2026-07-08T11:40:00+00:00,2026-07-08T12:10:00+00:00,NaN
3,2026-07-08T08:10:00+00:00,2026-07-08T08:34:00+00:00,9.0
7,2026-07-07T08:20:00+00:00,2026-07-07T08:43:00+00:00,18.0
16,2026-07-08T09:38:00+00:00,2026-07-08T09:44:00+00:00,NaN
18,2026-07-08T00:50:00+00:00,2026-07-08T01:20:00+00:00,NaN
19,2026-07-08T00:50:00+00:00,2026-07-08T01:20:00+00:00,NaN
20,2026-07-08T00:50:00+00:00,2026-07-08T01:20:00+00:00,NaN
21,2026-07-07T12:10:00+00:00,2026-07-07T12:21:00+00:00,NaN
23,2026-07-07T12:10:00+00:00,2026-07-07T12:21:00+00:00,NaN


In [7]:
df['dep_scheduled'] = pd.to_datetime(df['dep_scheduled'])
df['dep_actual'] = pd.to_datetime(df['dep_actual'])

df['actual_delay_min'] = (df['dep_actual'] - df['dep_scheduled']).dt.total_seconds() / 60

In [8]:
df[['dep_scheduled', 'dep_actual', 'dep_delay_min', 'actual_delay_min']].head(10)

,dep_scheduled,dep_actual,dep_delay_min,actual_delay_min
0,2026-07-09 02:40:00+00:00,NaT,NaN,NaN
1,2026-07-08 11:40:00+00:00,2026-07-08 12:10:00+00:00,NaN,30.0
2,2026-07-08 11:40:00+00:00,2026-07-08 12:10:00+00:00,NaN,30.0
3,2026-07-08 08:10:00+00:00,2026-07-08 08:34:00+00:00,9.0,24.0
4,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0,88.0
5,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0,88.0
6,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0,88.0
7,2026-07-07 08:20:00+00:00,2026-07-07 08:43:00+00:00,18.0,23.0
8,2026-07-07 08:20:00+00:00,NaT,NaN,NaN
9,2026-07-07 02:40:00+00:00,2026-07-07 03:22:00+00:00,41.0,42.0


In [9]:
df['arr_scheduled'] = pd.to_datetime(df['arr_scheduled'])
df['arr_actual'] = pd.to_datetime(df['arr_actual'])

df['actual_arr_delay_min'] = (df['arr_actual'] - df['arr_scheduled']).dt.total_seconds() / 60

In [10]:
df['actual_arr_delay_min'].notna().sum()

np.int64(57)

In [11]:
df['actual_delay_min'].notna().sum()

np.int64(109)

In [12]:
df.groupby('flight_status')['actual_delay_min'].apply(lambda x: x.notna().sum())

flight_status
active       24
landed       57
scheduled    28
Name: actual_delay_min, dtype: int64

In [13]:
df['has_departed'] = df['dep_actual'].notna()
df['has_arrived'] = df['arr_actual'].notna()

df[['flight_status', 'has_departed', 'has_arrived']].groupby('flight_status').sum()

,has_departed,has_arrived
flight_status,,
active,24,0
landed,57,57
scheduled,28,0


In [14]:
complete_flights = df[df['has_arrived'] == True]
complete_flights.shape

(57, 15)